In [1]:
import pandas as pd
import plotly.graph_objects as go

def plot_prosperity_day(day_str, product='TOMATOES'):
    """
    Loads data for a specific day and plots the interactive candlestick chart.
    """
    # 1. Load the data for the requested day
    # Note: Adjust sep=';' if your CSVs are comma-separated instead.
    df_prices = pd.read_csv(f'prices_round_0_day_{day_str}.csv') 
    df_trades = pd.read_csv(f'trades_round_0_day_{day_str}.csv', sep=';')

    # Filter datasets for the chosen product
    p_df = df_prices[df_prices['product'] == product].copy()
    t_df = df_trades[df_trades['symbol'] == product].copy()

    # 2. Aggregate into 500-timestamp candles for the Mid-Price
    candle_size = 500
    p_df['candle_bin'] = (p_df['timestamp'] // candle_size) * candle_size

    # Calculate Open, High, Low, Close (OHLC) for the candles
    ohlc = p_df.groupby('candle_bin').agg(
        Open=('mid_price', 'first'),
        High=('mid_price', 'max'),
        Low=('mid_price', 'min'),
        Close=('mid_price', 'last')
    ).reset_index()

    # 3. Build the Plotly Figure
    fig = go.Figure()

    # Add Candlesticks (Mid Price)
    fig.add_trace(go.Candlestick(
        x=ohlc['candle_bin'],
        open=ohlc['Open'],
        high=ohlc['High'],
        low=ohlc['Low'],
        close=ohlc['Close'],
        name='Mid Price',
        increasing_line_color='#26a69a', # Teal
        decreasing_line_color='#ef5350'  # Red
    ))

    # Add Best Bid Line
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'],
        y=p_df['bid_price_1'],
        mode='lines',
        line=dict(color='rgba(50, 100, 250, 0.4)', width=1), # Faint blue
        name='Best Bid',
        hoverinfo='skip' 
    ))

    # Add Best Ask Line
    fig.add_trace(go.Scatter(
        x=p_df['timestamp'],
        y=p_df['ask_price_1'],
        mode='lines',
        line=dict(color='rgba(250, 50, 50, 0.4)', width=1), # Faint red
        name='Best Ask',
        hoverinfo='skip'
    ))

    # Add Trade Markers
    if not t_df.empty:
        fig.add_trace(go.Scatter(
            x=t_df['timestamp'],
            y=t_df['price'],
            mode='markers',
            marker=dict(color='#ffca28', size=6, symbol='circle'), # Yellow dots
            name='Trades'
        ))

    # 4. Format Layout for Interactivity and Dark Theme
    fig.update_layout(
        title=f'{product} - Day {day_str} ({candle_size}-timestamp candles)',
        xaxis_title='Timestamp',
        yaxis_title='Price (XIRECs)',
        template='plotly_dark', 
        xaxis_rangeslider_visible=True, # Adjustable range slider
        hovermode='x unified', 
        height=700
    )

    # Display the interactive chart
    fig.show(renderer="browser")

# --- RUN THE VISUALIZATIONS ---

# This will render the chart for Day -1
plot_prosperity_day("-1", "TOMATOES")

# This will render a completely separate chart for Day -2 below it
plot_prosperity_day("-2", "TOMATOES")

In [5]:
import pandas as pd
import plotly.graph_objects as go

def plot_prosperity_day(day_str, product='TOMATOES'):
    """
    Loads data for a specific day and plots the interactive candlestick chart
    with all 3 levels of order book depth without cluttering the view.
    """
    # 1. Load the data for the requested day
    df_prices = pd.read_csv(f'prices_round_0_day_{day_str}.csv') 
    df_trades = pd.read_csv(f'trades_round_0_day_{day_str}.csv', sep=';')

    # Filter datasets for the chosen product
    p_df = df_prices[df_prices['product'] == product].copy()
    t_df = df_trades[df_trades['symbol'] == product].copy()

    # 2. Aggregate into 500-timestamp candles for the Mid-Price
    candle_size = 500
    p_df['candle_bin'] = (p_df['timestamp'] // candle_size) * candle_size

    ohlc = p_df.groupby('candle_bin').agg(
        Open=('mid_price', 'first'),
        High=('mid_price', 'max'),
        Low=('mid_price', 'min'),
        Close=('mid_price', 'last')
    ).reset_index()

    fig = go.Figure()

    # -------------------------------------------------------------
    # 3. ADD ORDER BOOK DEPTH (Plotted first to stay in background)
    # -------------------------------------------------------------
    
    # BIDS - Fading blue lines for depth (Opacity: 50% -> 25% -> 10%)
    bid_colors = ['rgba(50, 100, 250, 1)', 'rgba(50, 100, 250, 0.75)', 'rgba(50, 100, 250, 0.5)']
    for i in range(1, 4):
        col_name = f'bid_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=bid_colors[i-1], width=1 if i==1 else 0.5), # Thinner lines for L2 & L3
                name=f'Bid L{i}',
                hoverinfo='skip' 
            ))

    # ASKS - Fading red lines for depth (Opacity: 50% -> 25% -> 10%)
    ask_colors = ['rgba(250, 50, 50, 1)', 'rgba(250, 50, 50, 0.75)', 'rgba(250, 50, 50, 0.5)']
    for i in range(1, 4):
        col_name = f'ask_price_{i}'
        if col_name in p_df.columns:
            fig.add_trace(go.Scatter(
                x=p_df['timestamp'],
                y=p_df[col_name],
                mode='lines',
                line=dict(color=ask_colors[i-1], width=1 if i==1 else 0.5), # Thinner lines for L2 & L3
                name=f'Ask L{i}',
                hoverinfo='skip'
            ))

    # -------------------------------------------------------------
    # 4. ADD FOREGROUND ELEMENTS (Candlesticks & Trades)
    # -------------------------------------------------------------
    
    # Add Candlesticks (Mid Price)
    fig.add_trace(go.Candlestick(
        x=ohlc['candle_bin'],
        open=ohlc['Open'],
        high=ohlc['High'],
        low=ohlc['Low'],
        close=ohlc['Close'],
        name='Mid Price',
        increasing_line_color="#1cbd6f", # Teal
        decreasing_line_color="#b11411"  # Red
    ))

    # Add Trade Markers
    if not t_df.empty:
        fig.add_trace(go.Scatter(
            x=t_df['timestamp'],
            y=t_df['price'],
            mode='markers',
            marker=dict(color='#ffca28', size=6, symbol='circle'), # Yellow dots
            name='Trades'
        ))

    # 5. Format Layout for Interactivity and Dark Theme
    fig.update_layout(
        title=f'{product} - Day {day_str} ({candle_size}-timestamp candles) w/ Order Depth',
        xaxis_title='Timestamp',
        yaxis_title='Price (XIRECs)',
        template='plotly_dark', 
        xaxis_rangeslider_visible=True, # Adjustable range slider
        hovermode='x unified', 
        height=700
    )

    # Display the interactive chart in the browser
    fig.show(renderer="browser")

# --- RUN THE VISUALIZATIONS ---

# This will render the chart for Day -1
plot_prosperity_day("-1", "TOMATOES")

# This will render a completely separate chart for Day -2
plot_prosperity_day("-2", "TOMATOES")